In [141]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

### Reading the climate datasets, ibge and soil

In [123]:
%store -r df_era5_final
%store -r df_nasa_final
%store -r df_inmet_final
%store -r df_posology_final
%store -r df_ibge_final

In [87]:
def combine_dataset(ibge, soil, climate):
    
    ibge_climate = pd.merge(
        ibge,
        climate,
        on=['code', 'year'],
        how='inner'
    )

    ibge_climate_soil = pd.merge(
        ibge_climate,
        soil,
        on='code',
        how='inner'
    )

    ibge_climate_soil['corn_dencity_pct'] = (ibge_climate_soil['planted_area_ha'] / ibge_climate_soil['good_aptitude_ha']) * 100

    ibge_climate_soil = ibge_climate_soil[[col for col in ibge_climate_soil.columns if col not in ['planted_area_ha', 'city_x', 'city_y', 'good_aptitude_ha']]]
    
    return ibge_climate_soil

In [101]:
def split_scale_dataset(df, target, features):

    dados_treino = df[(df['year'] >= 2003) & (df['year'] <= 2020)]

    x_train = dados_treino[features]
    y_train = dados_treino[target]

    dados_teste = df[(df['year'] >= 2021) & (df['year'] <= 2024)]
    x_test = dados_teste[features]
    y_test = dados_teste[target]

    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    return {"x_train_scaled": x_train_scaled, "y_train": y_train, "x_test_scaled": x_test_scaled, "y_test": y_test}

In [85]:
def apply_model(splited_df, model, parameters):

    model_gs = GridSearchCV(model, parameters, scoring='neg_mean_squared_error', cv=10, return_train_score=True, verbose=1, n_jobs=-1)

    model_gs.fit(splited_df["x_train_scaled"], splited_df['y_train'])

    predictions = model_gs.predict(splited_df['x_test_scaled'])

    mae = mean_absolute_error(splited_df['y_test'], predictions)

    rmse = np.sqrt(mean_squared_error(splited_df['y_test'], predictions))

    r2 = r2_score(splited_df['y_test'], predictions)

    return {"mae": mae, "rmse": rmse, "r2": r2, "model_gs": model_gs}

### Combining the datasets

In [124]:
df_era5_combined = combine_dataset(df_ibge_final, df_posology_final, df_era5_final)
df_nasa_combined = combine_dataset(df_ibge_final, df_posology_final, df_nasa_final)
df_inmet_combined = combine_dataset(df_ibge_final, df_posology_final, df_inmet_final)

### Defining the models to be used

In [142]:
models_info = {
    "rfst": {
        "model": RandomForestRegressor(random_state=42), 
        "parameters": {'n_estimators': np.arange(100, 501, 100), 'max_depth': np.arange(1,30,4)}
    },
    "xgb": {
        "model": XGBRegressor(random_state=42),
        "parameters": {'n_estimators': np.arange(100, 501, 100), 'max_depth': np.arange(1,30,4), 'learning_rate': [0.1]}
    },
    "mlp": {
     "model": MLPRegressor(random_state=42, max_iter=500),
     "parameters": {'solver': ['lbfgs', 'adam'], 'hidden_layer_sizes': [(50,), (100,), (50, 25), (20, 10)],'activation': ['relu', 'tanh'], 'alpha': [0.001, 0.01, 0.1]}
    }
}

### Defining the datasets that will be used

In [143]:
target = 'yield_kg_ha'
soil_columns = ['corn_dencity_pct']
not_features = ['code', 'name', 'year', target]

datasets = {
    "era5_without_soil": (df_era5_combined, target, [col for col in df_era5_combined.columns if col not in soil_columns + not_features]),
    "nasa_without_soil": (df_nasa_combined, target, [col for col in df_nasa_combined.columns if col not in soil_columns + not_features]),
    "inmet_without_soil": (df_inmet_combined, target, [col for col in df_inmet_combined.columns if col not in soil_columns + not_features]),
    "era5_with_soil": (df_era5_combined, target, [col for col in df_era5_combined.columns if col not in not_features]),
    "nasa_with_soil": (df_nasa_combined, target, [col for col in df_nasa_combined.columns if col not in not_features]),
    "inmet_with_soil": (df_inmet_combined, target, [col for col in df_inmet_combined.columns if col not in not_features]),
}

### Applying the models

In [144]:
results = {}
for df_name, dataset in datasets.items():
    print('---------------------------------------')
    print(f"Dataset {df_name.upper()}, target {dataset[1]}, features {dataset[2]}.")
    df = split_scale_dataset(dataset[0], dataset[1], dataset[2])
    for model_name, model_param in models_info.items():
        print(f"Applying the model {model_name.upper()}.")
        results[f"{df_name}_{model_name}"] = {}
        results[f"{df_name}_{model_name}"]['model_results'] = apply_model(df, model_param['model'], model_param['parameters'])
        results[f"{df_name}_{model_name}"]['features'] = dataset[2]
        results[f"{df_name}_{model_name}"]['parameters'] = model_param['parameters']
        results[f"{df_name}_{model_name}"]['best_parans'] = results[f"{df_name}_{model_name}"]['model_results']['model_gs'].best_params_
        results[f"{df_name}_{model_name}"]['best_scores'] = results[f"{df_name}_{model_name}"]['model_results']['model_gs'].best_score_
        #print(results)
        print('---------------------------------------')

---------------------------------------
Dataset ERA5_WITHOUT_SOIL, target yield_kg_ha, features ['mean_temperature_c', 'max_temperature_c', 'min_temperature_c', 'total_rain_mm', 'sum_global_radiation_kj_m2', 'mean_relative_humidity_pct', 'max_relative_humidity_pct', 'min_relative_humidity_pct'].
Applying the model RFST.
Fitting 10 folds for each of 40 candidates, totalling 400 fits
---------------------------------------
Applying the model XGB.
Fitting 10 folds for each of 40 candidates, totalling 400 fits
---------------------------------------
Applying the model MLP.
Fitting 10 folds for each of 48 candidates, totalling 480 fits


c:\Users\franc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


---------------------------------------
---------------------------------------
Dataset NASA_WITHOUT_SOIL, target yield_kg_ha, features ['mean_temperature_c', 'max_temperature_c', 'min_temperature_c', 'total_rain_mm', 'sum_global_radiation_kj_m2', 'mean_relative_humidity_pct', 'max_relative_humidity_pct', 'min_relative_humidity_pct'].
Applying the model RFST.
Fitting 10 folds for each of 40 candidates, totalling 400 fits
---------------------------------------
Applying the model XGB.
Fitting 10 folds for each of 40 candidates, totalling 400 fits
---------------------------------------
Applying the model MLP.
Fitting 10 folds for each of 48 candidates, totalling 480 fits


c:\Users\franc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


---------------------------------------
---------------------------------------
Dataset INMET_WITHOUT_SOIL, target yield_kg_ha, features ['mean_temperature_c', 'max_temperature_c', 'min_temperature_c', 'total_rain_mm', 'sum_global_radiation_kj_m2', 'mean_relative_humidity_pct', 'max_relative_humidity_pct', 'min_relative_humidity_pct'].
Applying the model RFST.
Fitting 10 folds for each of 40 candidates, totalling 400 fits
---------------------------------------
Applying the model XGB.
Fitting 10 folds for each of 40 candidates, totalling 400 fits
---------------------------------------
Applying the model MLP.
Fitting 10 folds for each of 48 candidates, totalling 480 fits


...

c:\Users\franc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


---------------------------------------
---------------------------------------
Dataset NASA_WITH_SOIL, target yield_kg_ha, features ['mean_temperature_c', 'max_temperature_c', 'min_temperature_c', 'total_rain_mm', 'sum_global_radiation_kj_m2', 'mean_relative_humidity_pct', 'max_relative_humidity_pct', 'min_relative_humidity_pct', 'corn_dencity_pct'].
Applying the model RFST.
Fitting 10 folds for each of 40 candidates, totalling 400 fits
---------------------------------------
Applying the model XGB.
Fitting 10 folds for each of 40 candidates, totalling 400 fits
---------------------------------------
Applying the model MLP.
Fitting 10 folds for each of 48 candidates, totalling 480 fits


c:\Users\franc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


---------------------------------------
---------------------------------------
Dataset INMET_WITH_SOIL, target yield_kg_ha, features ['mean_temperature_c', 'max_temperature_c', 'min_temperature_c', 'total_rain_mm', 'sum_global_radiation_kj_m2', 'mean_relative_humidity_pct', 'max_relative_humidity_pct', 'min_relative_humidity_pct', 'corn_dencity_pct'].
Applying the model RFST.
Fitting 10 folds for each of 40 candidates, totalling 400 fits
---------------------------------------
Applying the model XGB.
Fitting 10 folds for each of 40 candidates, totalling 400 fits
---------------------------------------
Applying the model MLP.
Fitting 10 folds for each of 48 candidates, totalling 480 fits
---------------------------------------


c:\Users\franc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [145]:
results

{'era5_without_soil_rfst': {'model_results': {'mae': 1020.9152002168761,
   'rmse': np.float64(1195.0291946930413),
   'r2': -1.6157811637142787,
   'model_gs': GridSearchCV(cv=10, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
                param_grid={'max_depth': array([ 1,  5,  9, 13, 17, 21, 25, 29]),
                            'n_estimators': array([100, 200, 300, 400, 500])},
                return_train_score=True, scoring='neg_mean_squared_error',
                verbose=1)},
  'features': ['mean_temperature_c',
   'max_temperature_c',
   'min_temperature_c',
   'total_rain_mm',
   'sum_global_radiation_kj_m2',
   'mean_relative_humidity_pct',
   'max_relative_humidity_pct',
   'min_relative_humidity_pct'],
  'parameters': {'n_estimators': array([100, 200, 300, 400, 500]),
   'max_depth': array([ 1,  5,  9, 13, 17, 21, 25, 29])},
  'best_parans': {'max_depth': np.int64(9), 'n_estimators': np.int64(300)},
  'best_scores': np.float64(-1569813.5062383676)},
 'era